# Remotion Colab Render Workflow
這份筆記本專為您的 Remotion 專案設計，支援：
- 自動掛載 Google Drive
- 修正 Colab 上 `eslint-config-remotion` 的依賴衝突
- 自動建立所需的素材資料夾 (`drive_assets`)
- 輸出相容 Windows Media Player 的 yuv420p 影片
- 自動複製成果至 Google Drive

In [7]:
# Cell 0: 掛載 Drive 與 Clone 專案
from google.colab import drive
import os

drive.mount('/content/drive')

if not os.path.exists('/content/remotion'):
    !git clone https://github.com/Allen930311/remotion.git /content/remotion
else:
    %cd /content/remotion
    !git pull origin main

%cd /content/remotion

# 確保 drive_assets 選項目錄存在 (根據您的 auto-skill 經驗回報)
drive_assets_path = '/content/drive/MyDrive/remotion_assets'
os.makedirs(drive_assets_path, exist_ok=True)
os.makedirs(os.path.join(drive_assets_path, '02_Audio'), exist_ok=True)
os.makedirs(os.path.join(drive_assets_path, '03_Images'), exist_ok=True)
print("環境與專案取得完成！")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/remotion
From https://github.com/Allen930311/remotion
 * branch            main       -> FETCH_HEAD
Already up to date.
/content/remotion
環境與專案取得完成！


In [8]:
# Cell 1: 依賴修復與安裝
%cd /content/remotion
import json

package_json_path = 'package.json'
with open(package_json_path, 'r', encoding='utf-8') as f:
    pkg = json.load(f)

# 移除 eslint-config-remotion 避免在 Colab 報錯
if 'devDependencies' in pkg and 'eslint-config-remotion' in pkg['devDependencies']:
    del pkg['devDependencies']['eslint-config-remotion']
    with open(package_json_path, 'w', encoding='utf-8') as f:
        json.dump(pkg, f, indent=2)
    print("[依賴修復] 已移除 eslint-config-remotion")

!npm install

/content/remotion
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
up to date, audited 345 packages in 2s
⠧
⠧54 packages are looking for funding
⠧  run `npm fund` for details
⠧
8 high severity vulnerabilities

To address issues that do not require attention, run:
  npm audit fix

Some issues need review, and may require choosing
a different dependency.

Run `npm audit` for details.
⠧

In [9]:
# Cell 2: 渲染設定與執行
%cd /content/remotion
import os
import shutil

# 設定環境變數 (如需更改參數請修改此處)
os.environ["ENTRY_FILE"] = "src/index.ts"  # Remotion 進入點檔案
os.environ["COMP_NAME"] = "GeopoliticsExplainer"         # Composition 名稱
os.environ["OUTPUT_FILE"] = "out/geopolitics_explainer.mp4"

# 在 Colab 渲染時，建議使用 npx remotion render 搭配相容性參數
# 確保影片能在 Windows 預設播放器正常播放 (-pixel-format yuv420p, -codec h264)
render_cmd = f"npx remotion render {os.environ['ENTRY_FILE']} {os.environ['COMP_NAME']} {os.environ['OUTPUT_FILE']} --pixel-format=yuv420p --codec=h264 --concurrency=2"
print(f"Executing: {render_cmd}")

!{render_cmd}

# 將成品複製到 Google Drive
output_path = f"/content/remotion/{os.environ['OUTPUT_FILE']}"
drive_dest = f"/content/drive/MyDrive/remotion_assets/video_rendered.mp4"

if os.path.exists(output_path):
    shutil.copy(output_path, drive_dest)
    print(f"\n✅ 影片已成功輸出並複製至 Drive: {drive_dest}")
else:
    print("\n❌ 找不到輸出的影片，渲染可能失敗。")

/content/remotion
Executing: npx remotion render src/index.ts GeopoliticsExplainer out/geopolitics_explainer.mp4 --pixel-format=yuv420p --codec=h264 --concurrency=2
⠙Bundling code        ━╺━━━━━━━━━━━━━━━━ 6%Bundling code        ━━╺━━━━━━━━━━━━━━━ 12%Bundling code        ━━━━━━━╺━━━━━━━━━━ 41%Bundling code        ━━━━━━━━━━━╸━━━━━━ 65%Bundling code        ━━━━━━━━━━━━╸━━━━━ 71%Bundling code        ━━━━━━━━━━━━━╸━━━━ 76%Bundling code        ━━━━━━━━━━━━━━╸━━━ 81%Bundling code        ━━━━━━━━━━━━━━━╸━━ 87%Bundling code        ━━━━━━━━━━━━━━━━╸━ 92%Bundling code        ━━━━━━━━━━━━━━━━━╸ 98%Bundling code        ━━━━━━━━━━━━━━━━━━ 100%Bundled code         ━━━━━━━━━━━━━━━━━━ 4956ms

Symbolicating minified error message...
Error: Could not find composition with ID GeopoliticsExplainer. Available compositions: MyComp, ColabShowcase, AutoSkillIntro, SkillsShowcase, AnimationPlayground, CryptoPaymentAnimationAn error occurred:
 Error  Error: Could not find composition with ID GeopoliticsExplain